In [1]:
import sys 

assert sys.version_info >= (3,10)

In [2]:
from packaging.version import Version
import torch

assert Version(torch.__version__) >= Version("2.6.0")

In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

In [4]:
import matplotlib.pyplot as plt

plt.rc("font", size=14)
plt.rc("legend", fontsize=14)
plt.rc("axes", labelsize=14, titlesize=14)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)

In [5]:
import deepxde as dde
import numpy as np

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Setting up the backend


In [6]:
dde.config.set_default_float("float64")
print(f"Backend: {dde.backend.backend_name}")


Set the default float type to float64
Backend: pytorch


Exact solution for validation

In [7]:
def exact_solution(x):
    return (x + 1) ** 2

Domain geometry

In [8]:
geom = dde.geometry.Interval(-1, 1)

Define the Left and Right Boundary Conditions

In [9]:
def pde(x, y):
    dy_xx = dde.grad.hessian(y, x, i=0, j=0)
    return dy_xx - 2

def boundary_left(x, on_boundary):
    return on_boundary and np.isclose(x[0], -1)

bc_left = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_left)

def boundary_right(x, on_boudary):
    return on_boudary and np.isclose(x[0], 1)

bc_right = dde.icbc.NeumannBC(geom, lambda x: 4, boundary_right)

Creating data using numpy

In [10]:
observe_x  = np.linspace(-1, 1, 35).reshape(-1, 1)
observe_y = exact_solution(observe_x)
noise = 0.1 * np.random.randn(35, 1)
observe_y = observe_y + noise
observe = dde.icbc.PointSetBC(observe_x, observe_y, component=0)


Combine all data

In [12]:
data = dde.data.PDE(
    geom, 
    pde,
    [bc_left, bc_right, observe],
    num_domain=3000,
    num_boundary=200,
    num_test=500
)

Build a Neural Network and create a Model

In [13]:
net = dde.nn.FNN([1, 256, 128, 64, 1],"tanh", "Glorot uniform")

model = dde.Model(data, net)